<a href="https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:
Prioritize content that shows a clear opportunity for improvement based on recent performance.

The baseline score combines:
- Current organic clicks
- Search impressions
- CTR weakness (high impressions but lower clicks)
- Recent performance trend

Reason codes:
- HIGH_IMPRESSIONS_LOW_CTR: Content gets visibility but needs better click performance.
- TRAFFIC_DECLINE: Content shows decreasing recent clicks.
- GROWTH_OPPORTUNITY: Content has enough impressions and ranking potential.
- STABLE_PERFORMER: Content is already performing consistently.

In [12]:
!pip -q install duckdb huggingface_hub

In [13]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass(
    'Paste your Hugging Face READ token: '
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully")

Paste your Hugging Face READ token: ··········
Connected successfully


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check available variables
print([x for x in globals().keys() if not x.startswith("_")])

['In', 'Out', 'get_ipython', 'exit', 'quit', 'os', 'getpass', 'duckdb', 'HF_TOKEN', 'con', 'REL', 'TABLES', 'pd']


In [15]:
print(TABLES)

{'dim_clients': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')", 'dim_content': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')", 'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_daily_sample': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


In [16]:
import pandas as pd
import os

# Load daily performance sample
df = con.sql(f"""
SELECT *
FROM {TABLES['fact_daily_sample']}
LIMIT 200000
""").df()

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(200000, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
import numpy as np
import os

# Date
df["report_date"] = pd.to_datetime(df["report_date"])


# CTR
df["ctr"] = df["gsc_clicks"] / (df["gsc_impressions"] + 1)


# Score
df["score"] = (
    df["gsc_impressions"].rank(pct=True) * 0.4 +
    (1 - df["ctr"].rank(pct=True)) * 0.4 +
    (1 - df["gsc_sum_position"].rank(pct=True)) * 0.2
)


# Calculate thresholds once
high_imp = df["gsc_impressions"].quantile(0.75)
median_ctr = df["ctr"].median()
median_imp = df["gsc_impressions"].median()


# Vectorized reason codes
df["reason_code"] = np.select(
    [
        (df["gsc_impressions"] > high_imp) &
        (df["ctr"] < median_ctr),

        df["gsc_sum_position"] > 10,

        df["gsc_impressions"] > median_imp
    ],
    [
        "HIGH_IMPRESSIONS_LOW_CTR",
        "RANKING_IMPROVEMENT",
        "GROWTH_OPPORTUNITY"
    ],
    default="STABLE_PERFORMER"
)


# Rank
df = df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1


# Save
os.makedirs("work/outputs", exist_ok=True)

df.head(100).to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


print("Completed")
print(df.shape)

df.head(10)

Completed
(200000, 35)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,ctr,score,reason_code,rank
0,2026-06-03,client_62f4a7e64f5e0096,content_031934b7a288cc11,True,False,True,<NA>,25,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.698637,GROWTH_OPPORTUNITY,1
1,2026-06-01,client_62f4a7e64f5e0096,content_01a33ec5c454390d,True,False,True,<NA>,15,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.687238,GROWTH_OPPORTUNITY,2
2,2026-06-03,client_62f4a7e64f5e0096,content_7147470d51a4a94c,True,False,True,<NA>,11,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.680121,GROWTH_OPPORTUNITY,3
3,2026-06-03,client_62f4a7e64f5e0096,content_7bbaba07e1765928,True,False,True,<NA>,11,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.680121,GROWTH_OPPORTUNITY,4
4,2026-06-03,client_62f4a7e64f5e0096,content_0dac238195631de4,True,False,True,<NA>,8,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.672386,GROWTH_OPPORTUNITY,5
5,2026-06-02,client_23a62021009f63c4,content_464a58519b2c445c,True,True,True,True,8,0,0,...,0,0,0,0,0,2026-06,0.0,0.672386,GROWTH_OPPORTUNITY,6
6,2026-06-03,client_62f4a7e64f5e0096,content_b0f62c76d21a801a,True,False,True,<NA>,8,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.672386,GROWTH_OPPORTUNITY,7
7,2026-06-01,client_62f4a7e64f5e0096,content_ce265ddc23a9637c,True,False,True,<NA>,7,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.669040,GROWTH_OPPORTUNITY,8
8,2026-06-03,client_62f4a7e64f5e0096,content_97fd8c04f16c9550,True,False,True,<NA>,7,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.669040,GROWTH_OPPORTUNITY,9
9,2026-06-02,client_23a62021009f63c4,content_72a6c9f9adae0331,True,True,True,False,7,0,0,...,0,0,0,0,0,2026-06,0.0,0.669040,GROWTH_OPPORTUNITY,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df.head(20).copy()


def action(reason):
    if reason == "HIGH_IMPRESSIONS_LOW_CTR":
        return "Improve title/meta description and search snippet"

    elif reason == "RANKING_IMPROVEMENT":
        return "Optimize content to improve ranking"

    elif reason == "GROWTH_OPPORTUNITY":
        return "Expand content and target related queries"

    else:
        return "Monitor performance"


top20["action"] = top20["reason_code"].apply(action)

top20["confidence_note"] = (
    "Medium confidence based on observed impressions, clicks and ranking signals"
)

top20["what_would_make_it_wrong"] = (
    "Could be incorrect if seasonality, tracking gaps, or business priorities differ"
)


top20.to_csv(
    "work/outputs/top20_review.csv",
    index=False
)


top20

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_other,scroll_events,month,ctr,score,reason_code,rank,action,confidence_note,what_would_make_it_wrong
0,2026-06-03,client_62f4a7e64f5e0096,content_031934b7a288cc11,True,False,True,<NA>,25,0,0,...,<NA>,<NA>,2026-06,0.0,0.698637,GROWTH_OPPORTUNITY,1,Expand content and target related queries,Medium confidence based on observed impression...,"Could be incorrect if seasonality, tracking ga..."
1,2026-06-01,client_62f4a7e64f5e0096,content_01a33ec5c454390d,True,False,True,<NA>,15,0,0,...,<NA>,<NA>,2026-06,0.0,0.687238,GROWTH_OPPORTUNITY,2,Expand content and target related queries,Medium confidence based on observed impression...,"Could be incorrect if seasonality, tracking ga..."
2,2026-06-03,client_62f4a7e64f5e0096,content_7147470d51a4a94c,True,False,True,<NA>,11,0,0,...,<NA>,<NA>,2026-06,0.0,0.680121,GROWTH_OPPORTUNITY,3,Expand content and target related queries,Medium confidence based on observed impression...,"Could be incorrect if seasonality, tracking ga..."
3,2026-06-03,client_62f4a7e64f5e0096,content_7bbaba07e1765928,True,False,True,<NA>,11,0,0,...,<NA>,<NA>,2026-06,0.0,0.680121,GROWTH_OPPORTUNITY,4,Expand content and target related queries,Medium confidence based on observed impression...,"Could be incorrect if seasonality, tracking ga..."
4,2026-06-03,client_62f4a7e64f5e0096,content_0dac238195631de4,True,False,True,<NA>,8,0,0,...,<NA>,<NA>,2026-06,0.0,0.672386,GROWTH_OPPORTUNITY,5,Expand content and target related queries,Medium confidence based on observed impression...,"Could be incorrect if seasonality, tracking ga..."
5,2026-06-02,client_23a62021009f63c4,content_464a58519b2c445c,True,True,True,True,8,0,0,...,0,0,2026-06,0.0,0.672386,GROWTH_OPPORTUNITY,6,Expand content and target related queries,Medium confidence based on observed impression...,"Could be incorrect if seasonality, tracking ga..."
6,2026-06-03,client_62f4a7e64f5e0096,content_b0f62c76d21a801a,True,False,True,<NA>,8,0,0,...,<NA>,<NA>,2026-06,0.0,0.672386,GROWTH_OPPORTUNITY,7,Expand content and target related queries,Medium confidence based on observed impression...,"Could be incorrect if seasonality, tracking ga..."
7,2026-06-01,client_62f4a7e64f5e0096,content_ce265ddc23a9637c,True,False,True,<NA>,7,0,0,...,<NA>,<NA>,2026-06,0.0,0.669040,GROWTH_OPPORTUNITY,8,Expand content and target related queries,Medium confidence based on observed impression...,"Could be incorrect if seasonality, tracking ga..."
8,2026-06-03,client_62f4a7e64f5e0096,content_97fd8c04f16c9550,True,False,True,<NA>,7,0,0,...,<NA>,<NA>,2026-06,0.0,0.669040,GROWTH_OPPORTUNITY,9,Expand content and target related queries,Medium confidence based on observed impression...,"Could be incorrect if seasonality, tracking ga..."
9,2026-06-02,client_23a62021009f63c4,content_72a6c9f9adae0331,True,True,True,False,7,0,0,...,0,0,2026-06,0.0,0.669040,GROWTH_OPPORTUNITY,10,Expand content and target related queries,Medium confidence based on observed impression...,"Could be incorrect if seasonality, tracking ga..."


Weak picks:

Some recommendations may look weak because the baseline only uses observed search performance signals. Pages with high impressions but very low clicks may not always represent real opportunities because they could be affected by seasonality, brand searches, tracking issues, or business priorities.

Leakage check:

The scoring rule only uses available performance metrics from the dataset. No future windows, product flags, private queries, or outcome fields were used.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check weak picks from top-ranked items

weak_picks = df[
    (df["gsc_impressions"] == 0) |
    (df["gsc_clicks"] == 0)
].head(10)

print("Example weak picks:")
display(weak_picks)


# Leakage check

possible_leakage = [
    col for col in df.columns
    if any(word in col.lower() for word in [
        "future",
        "conversion",
        "revenue",
        "product_flag",
        "label",
        "target"
    ])
]


print("\nPossible leakage columns:")
print(possible_leakage)


# Confirm scoring columns

print("\nColumns used for scoring:")
print([
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_sum_position"
])


print("\nLeakage check complete")

Example weak picks:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,ctr,score,reason_code,rank
0,2026-06-03,client_62f4a7e64f5e0096,content_031934b7a288cc11,True,False,True,<NA>,25,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.698637,GROWTH_OPPORTUNITY,1
1,2026-06-01,client_62f4a7e64f5e0096,content_01a33ec5c454390d,True,False,True,<NA>,15,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.687238,GROWTH_OPPORTUNITY,2
2,2026-06-03,client_62f4a7e64f5e0096,content_7147470d51a4a94c,True,False,True,<NA>,11,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.680121,GROWTH_OPPORTUNITY,3
3,2026-06-03,client_62f4a7e64f5e0096,content_7bbaba07e1765928,True,False,True,<NA>,11,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.680121,GROWTH_OPPORTUNITY,4
4,2026-06-03,client_62f4a7e64f5e0096,content_0dac238195631de4,True,False,True,<NA>,8,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.672386,GROWTH_OPPORTUNITY,5
5,2026-06-02,client_23a62021009f63c4,content_464a58519b2c445c,True,True,True,True,8,0,0,...,0,0,0,0,0,2026-06,0.0,0.672386,GROWTH_OPPORTUNITY,6
6,2026-06-03,client_62f4a7e64f5e0096,content_b0f62c76d21a801a,True,False,True,<NA>,8,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.672386,GROWTH_OPPORTUNITY,7
7,2026-06-01,client_62f4a7e64f5e0096,content_ce265ddc23a9637c,True,False,True,<NA>,7,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.669040,GROWTH_OPPORTUNITY,8
8,2026-06-03,client_62f4a7e64f5e0096,content_97fd8c04f16c9550,True,False,True,<NA>,7,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06,0.0,0.669040,GROWTH_OPPORTUNITY,9
9,2026-06-02,client_23a62021009f63c4,content_72a6c9f9adae0331,True,True,True,False,7,0,0,...,0,0,0,0,0,2026-06,0.0,0.669040,GROWTH_OPPORTUNITY,10



Possible leakage columns:
[]

Columns used for scoring:
['gsc_impressions', 'gsc_clicks', 'ctr', 'gsc_sum_position']

Leakage check complete


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.